In [15]:
# !pip install h3

In [16]:
import numpy as np
import pandas as pd
import h3
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import lightgbm as lgb
import joblib
import warnings
warnings.filterwarnings("ignore")

In [18]:
from app.features.custom_transformers import *

ModuleNotFoundError: No module named 'app'

In [ ]:
class ColumnSelector(BaseEstimator, TransformerMixin):
    """Keep only numeric columns, drop 'id' and 'trip_duration'"""
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        cols_to_keep = [col for col in numeric_cols if col not in ['id', 'trip_duration']]
        return X[cols_to_keep].copy()


class EnhancedNYCFeatureEngineer(BaseEstimator, TransformerMixin):
    """All your original logic + improvements for better score"""
    def __init__(self, n_clusters=30, random_state=42):
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.kmeans = None
        self.pca = None
    
    def fit(self, X, y=None):
        X = X.copy()
        coords = np.vstack((
            X[['pickup_latitude', 'pickup_longitude']].values,
            X[['dropoff_latitude', 'dropoff_longitude']].values
        ))
        self.kmeans = MiniBatchKMeans(
            n_clusters=self.n_clusters, batch_size=10000,
            random_state=self.random_state, n_init=20
        ).fit(coords)
        
        self.pca = PCA(n_components=2, random_state=self.random_state)
        self.pca.fit(coords)
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # Original + improved datetime features
        X['pickup_datetime'] = pd.to_datetime(X['pickup_datetime'])
        X['pickup_hour'] = X['pickup_datetime'].dt.hour
        X['pickup_minute'] = X['pickup_datetime'].dt.minute
        X['pickup_day'] = X['pickup_datetime'].dt.dayofweek
        X['pickup_month'] = X['pickup_datetime'].dt.month
        X['is_weekend'] = X['pickup_day'].isin([5, 6]).astype(int)
        
        def get_part_of_day(hour):
            if 5 <= hour < 12: return "morning"
            elif 12 <= hour < 17: return "afternoon"
            elif 17 <= hour < 21: return "evening"
            else: return "night"
        X['part_of_day'] = X['pickup_hour'].apply(get_part_of_day)
        
        X['rush_hour'] = ((X['pickup_hour'].between(7,9)) | 
                         (X['pickup_hour'].between(16,19))).astype(int)
        X['is_night'] = X['pickup_hour'].isin([0,1,2,3,4,22,23]).astype(int)
        
        # Original distance features
        X["distance"] = X.apply(lambda row: h3.great_circle_distance(
            (row["pickup_latitude"], row["pickup_longitude"]),
            (row["dropoff_latitude"], row["dropoff_longitude"]), unit="km"), axis=1)
        
        X['manhattan_distance'] = (
            abs(X['pickup_latitude'] - X['dropoff_latitude']) +
            abs(X['pickup_longitude'] - X['dropoff_longitude'])
        )
        
        # Original bearing
        def calculate_bearing(lat1, lon1, lat2, lon2):
            lon_diff = np.radians(lon2 - lon1)
            lat1_r = np.radians(lat1)
            lat2_r = np.radians(lat2)
            x = np.sin(lon_diff) * np.cos(lat2_r)
            y = np.cos(lat1_r) * np.sin(lat2_r) - (np.sin(lat1_r) * np.cos(lat2_r) * np.cos(lon_diff))
            return np.degrees(np.arctan2(x, y))
        
        X['bearing'] = calculate_bearing(
            X['pickup_latitude'], X['pickup_longitude'],
            X['dropoff_latitude'], X['dropoff_longitude']
        )
        
        # Original clustering
        X['pickup_cluster'] = self.kmeans.predict(X[['pickup_latitude', 'pickup_longitude']].values)
        X['dropoff_cluster'] = self.kmeans.predict(X[['dropoff_latitude', 'dropoff_longitude']].values)
        
        # Extra strong features
        X['cluster_pair'] = X['pickup_cluster'].astype(str) + "_" + X['dropoff_cluster'].astype(str)
        
        # PCA
        coords_p = X[['pickup_latitude', 'pickup_longitude']].values
        coords_d = X[['dropoff_latitude', 'dropoff_longitude']].values
        X[['pca_pickup1', 'pca_pickup2']] = self.pca.transform(coords_p)
        X[['pca_dropoff1', 'pca_dropoff2']] = self.pca.transform(coords_d)
        
        # One-hot encoding
        X = pd.get_dummies(X, columns=['part_of_day', 'store_and_fwd_flag'], dtype=int)
        
        # Limited cluster pairs
        top_pairs = X['cluster_pair'].value_counts().head(40).index
        X['cluster_pair'] = X['cluster_pair'].apply(lambda x: x if x in top_pairs else 'other')
        X = pd.get_dummies(X, columns=['cluster_pair'], dtype=int, prefix='pair')
        
        # Drop raw columns (exactly your original logic)
        drop_cols = ['pickup_datetime', 'pickup_longitude','pickup_latitude',
                     'dropoff_longitude','dropoff_latitude', 'id']
        X.drop([col for col in drop_cols if col in X.columns], axis=1, inplace=True)
        
        return X


class LogTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        for col in ['distance', 'manhattan_distance', 'passenger_count']:
            if col in X.columns:
                X[col] = np.log1p(X[col].clip(lower=0))
        return X

In [ ]:
from sklearn.pipeline import Pipeline

def build_full_pipeline():
    """Returns a single Pipeline with preprocessing + LightGBM"""
    
    preprocessor = Pipeline([
        ('feature_engineer', EnhancedNYCFeatureEngineer(n_clusters=30)),
        ('log_transform', LogTransformer()),
        ('selector', ColumnSelector()),
        ('imputer', SimpleImputer(strategy='median'))
    ])
    
    # Final model step
    model = lgb.LGBMRegressor(
        n_estimators=2500,
        learning_rate=0.04,
        num_leaves=128,
        max_depth=12,
        subsample=0.85,
        colsample_bytree=0.80,
        min_child_samples=20,
        reg_alpha=0.15,
        reg_lambda=0.15,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    # Combine into one big pipeline
    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    return full_pipeline

In [ ]:
train_path = "/home/parth/Desktop/parth/Machine Learning/TASK 2/.venv/DATA/RAW/train.csv"
test_path = "/home/parth/Desktop/parth/Machine Learning/TASK 2/.venv/DATA/RAW/test.csv"

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df.drop(['dropoff_datetime'], axis=1, inplace=True, errors='ignore')
train_df = train_df[train_df['trip_duration'] <= 3600 * 24].copy()

# Stronger cleaning: remove obvious outliers
train_df = train_df[(train_df['trip_duration'] >= 10) & 
                    (train_df['trip_duration'] <= 3600*3)]
test_ids = test_df["id"].copy()
print("After cleaning train shape:", train_df.shape)

After cleaning train shape: (1454548, 10)


In [ ]:
train_df.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [ ]:
full_pipeline = build_full_pipeline()
X = train_df.drop(["trip_duration"], axis=1, errors='ignore')
y = np.log1p(train_df["trip_duration"].values)

In [ ]:
print("Fitting full pipeline on training data...")
full_pipeline.fit(X, y)

Fitting full pipeline on training data...


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineer', ...), ('log_transform', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,n_clusters,30
,random_state,42
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `

In [ ]:
train_df.head()

,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42)
val_pred = np.nan_to_num(full_pipeline.predict(X_val))

rmse = np.sqrt(mean_squared_error(y_val, val_pred))
mae = mean_absolute_error(y_val, val_pred)
r2 = r2_score(y_val, val_pred)

print("\nValidation Results:")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")


Validation Results:
RMSE : 0.3251
MAE  : 0.2397
R2   : 0.8138


In [ ]:
joblib.dump(full_pipeline, "taxi_full_pipeline.pkl")
print("\nFull pipeline saved as: taxi_full_pipeline.pkl")


Full pipeline saved as: taxi_full_pipeline.pkl


In [ ]:
print("Generating submission...")
test_pred_log = full_pipeline.predict(test_df)
test_pred = np.expm1(test_pred_log)

submission = pd.DataFrame({"id": test_ids, "trip_duration": test_pred})
submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")

Generating submission...
submission.csv created successfully!


In [ ]:
submission.head()

,id,trip_duration
0,id3004672,728.304452
1,id3505355,541.806673
2,id1217141,384.861912
3,id2150126,724.324597
4,id1598245,370.914056
